# Brain Tumor Segmentation using SwinUNETR

## Project Overview
This notebook implements a deep learning pipeline to segment brain tumors from MRI scans using the BraTS dataset. We utilize a SwinUNETR architecture to predict segmentation masks from multi-modal MRI inputs.

## Objectives
1.  **Data Loading**: Load 3D MRI volumes (NIfTI format).
2.  **Preprocessing**: Normalize and format data for the model.
3.  **Model**: Implement a SwinUNETR architecture in PyTorch.
4.  **Training**: Train the model using Dice Loss.
5.  **Visualization**: Visualize inputs, ground truth, and predictions.


## 1. Setup and Imports
Install necessary packages if not already present.


In [ ]:
!pip install monai nibabel matplotlib torch torchvision tqdm scikit-learn comet_ml -q

In [ ]:
import comet_ml
from comet_ml import Experiment
import os
import glob
import gc
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from monai import transforms
from monai import data

from monai.networks.nets import SwinUNETR, DynUNet
from monai.losses import DiceLoss
from monai.inferers import sliding_window_inference

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
print(f"Using device: {device}")

from torch.optim.lr_scheduler import CosineAnnealingLR
from monai.metrics import DiceMetric
from monai.data import decollate_batch
from monai.transforms import AsDiscrete, Compose, EnsureType, Activations


## 2. Data Configuration

In [ ]:
# === CONFIGURATION ===
# Initialize Comet ML Experiment
experiment = Experiment(
    api_key = "VDWJEDjieOQbqDskMAw7Cr1av",
    project_name="brain-tumor-segmentation",
    auto_metric_logging=True,
    auto_param_logging=True,
    auto_histogram_weight_logging=True,
    auto_histogram_gradient_logging=True,
    auto_histogram_activation_logging=True,
)

DATASET_ROOT = "/kaggle/input/brain-tumor-segmentation-hackathon"

# Parameters
IMG_SIZE = 128
BATCH_SIZE = 1
LEARNING_RATE = 1e-4
NUM_EPOCHS = 100
VALIDATION_SPLIT = 0.2

# Log parameters
parameters = {
    "IMG_SIZE":IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "epochs": NUM_EPOCHS,
    "validation_split": VALIDATION_SPLIT,
}
experiment.log_parameters(parameters)

test_index = 0      # Index for visualization

## 3. Dataset Implementation
We define a custom `BraTSDataset` class.
-   **Inputs**: Reads 4 modalites: FLAIR, T1, T1ce, T2.
-   **Labels**: 
    -   Label 1: Necrotic/core (NCR)
    -   Label 2: Edema (ED)
    -   Label 4: Enhancing Tumor (ET)
    -   *Mapped to 0, 1, 2, 3 for training*.


In [ ]:
class BraTSDataset(Dataset):
    def __init__(self, root_dir, transform=None, img_size=128, mode='train', test_dir=None):
        self.root_dir = root_dir
        self.transform = transform
        self.img_size = img_size
        self.mode = mode
        
        if mode == 'test':
            self.case_folders = sorted(glob.glob(os.path.join(test_dir, "BraTS2021_*")))
            print(f"Found {len(self.case_folders)} cases for test")
        else:
            # Get all case folders
            self.case_folders = sorted(glob.glob(os.path.join(root_dir, "BraTS2021_*")))
                    
            # Simple train/val split based on index
            split_idx = int(len(self.case_folders) * (1 - VALIDATION_SPLIT))
            if mode == 'train':
                self.case_folders = self.case_folders[:split_idx]
            else:
                self.case_folders = self.case_folders[split_idx:]
            
            print(f"Found {len(self.case_folders)} cases for {mode}")
        
        # Pre-load file paths
        self.data_list = []
        print(f"Pre-loading file paths for {mode}...")
        for case_path in tqdm(self.case_folders):
            case_id = os.path.basename(case_path)
            try:
                flair_path = self._get_file_path(case_path, case_id, "flair")
                t1_path = self._get_file_path(case_path, case_id, "t1")
                t1ce_path = self._get_file_path(case_path, case_id, "t1ce")
                t2_path = self._get_file_path(case_path, case_id, "t2")
                
                item = {
                    "image": [flair_path, t1_path, t1ce_path, t2_path],
                    "id": case_id
                }
                
                if mode != 'test':
                    seg_path = self._get_file_path(case_path, case_id, "seg")
                    item["label"] = seg_path
                
                self.data_list.append(item)
                
            except Exception as e:
                print(f"Skipping case {case_id}: {e}")
    
    def __len__(self):
        return len(self.data_list)

    def _get_file_path(self, case_path, case_id, modality):
        # Helper to find the .nii file for a given modality.
        # Structure: Case/Case_modality.nii/file.nii
        
        folder_name = f"{case_id}_{modality}.nii"
        folder_path = os.path.join(case_path, folder_name)
        
        # 1. Try to find the file in the specific folder (Nested structure)
        if os.path.isdir(folder_path):
            files = glob.glob(os.path.join(folder_path, "*.nii"))
            if files:
                if len(files) > 1:
                    print(f"Multiple files found in {folder_path}: {len(files)}")
                return files[0]
                
        # 2. Fallback: Check if the folder_path itself is actually the file
        if os.path.isfile(folder_path):
            return folder_path

        # 3. Final Fallback: If not found, search the parent case_path for all available .nii files
        available_files = glob.glob(os.path.join(case_path, "**", "*.nii"), recursive=True)
      
        raise FileNotFoundError(f"File for modality '{modality}' not found in {case_path}")

    def __getitem__(self, idx):
        data = self.data_list[idx]
        if self.transform:
            data = self.transform(data)
        return data


In [ ]:
def get_loader(batch_size, data_dir, roi):
    # Initialize datasets
    train_files = BraTSDataset(data_dir, mode='train')
    validation_files = BraTSDataset(data_dir, mode='val')
    train_transform = transforms.Compose(
        [
            transforms.LoadImaged(keys=["image", "label"]),
            transforms.EnsureChannelFirstd(keys=["image"]),
            transforms.ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
            transforms.CropForegroundd(
                keys=["image", "label"],
                source_key="image",
                k_divisible=[roi[0], roi[1], roi[2]],
                allow_smaller=True,
            ),
            transforms.RandSpatialCropd(
                keys=["image", "label"],
                roi_size=[roi[0], roi[1], roi[2]],
                random_size=False,
            ),
            transforms.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
            transforms.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
            transforms.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
            transforms.NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
            transforms.RandScaleIntensityd(keys="image", factors=0.1, prob=1.0),
            transforms.RandShiftIntensityd(keys="image", offsets=0.1, prob=1.0),
        ]
    )
    val_transform = transforms.Compose( 
        [
            transforms.LoadImaged(keys=["image", "label"]),
            transforms.EnsureChannelFirstd(keys=["image"]),
            transforms.ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
            transforms.NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
        ]
    )

    train_ds = data.Dataset(data=train_files, transform=train_transform)

    train_loader = data.DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
    )
    val_ds = data.Dataset(data=validation_files, transform=val_transform)
    val_loader = data.DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
    )

    return train_loader, val_loader

## 5. Training


In [ ]:
# Initialize Model
model = SwinUNETR(
    in_channels=4,
    out_channels=3,
    feature_size=48,
    drop_rate=0.0,
    attn_drop_rate=0.0,
    dropout_path_rate=0.0,
    use_checkpoint=True,
).to(device)
criterion = DiceLoss(sigmoid=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print("Model and Loaders initialized.")

In [ ]:
import requests

# The Comet.ml asset URL
url = "https://www.comet.com/api/asset/download?assetId=0c622956588542b9a81c4aa2a5537f8e&experimentKey=2246cbc310dc4a5cafaad0e48aedf398"

# Define the local filename (change the extension based on your model type: .pt, .h5, .pkl)
model_path = "downloaded_model.pt" 

print("Downloading model...")
response = requests.get(url, stream=True)

if response.status_code == 200:
    with open(model_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"Model downloaded successfully and saved to {model_path}")
else:
    print(f"Failed to download. Status code: {response.status_code}")

In [ ]:
# Enhanced Training function with Comet ML Logging and DiceMetric
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, patience=5, val_interval=1, save_path='best_model.pth'):
    best_val_dice = -1.0
    epochs_no_improve = 0
    scaler = torch.amp.GradScaler('cuda') # Initialize GradScaler for AMP
    
    # Initialize Metrics
    dice_metric = DiceMetric(include_background=True, reduction="mean")
    dice_metric_batch = DiceMetric(include_background=True, reduction="mean_batch")
    
    # Post-processing transforms for validation
    post_trans = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
    
    with experiment.train():
        step = 0
        for epoch in range(num_epochs):
            # === TRAINING ===
            model.train()
            train_loss = 0
            loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
            
            for batch_data in loop:
                images = batch_data["image"].to(device)
                masks = batch_data["label"].to(device)
                
                # Forward with AMP
                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, masks)
                
                # Backward
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                
                train_loss += loss.item()
                loop.set_postfix(loss=loss.item())
                
                # Log batch loss
                experiment.log_metric("batch_train_loss", loss.item(), step=step)
                step += 1
                
            avg_train_loss = train_loss / len(train_loader)
            experiment.log_metric("epoch_train_loss", avg_train_loss, step=epoch)
            
            # Optimization: Clear memory
            del images, masks, outputs, loss
            gc.collect()
            torch.cuda.empty_cache()
            
            # === VALIDATION ===
            if (epoch + 1) % val_interval == 0:
                model.eval()
                val_loss = 0
                with torch.no_grad():
                    with experiment.validate():
                        for batch_data in val_loader:
                            images = batch_data["image"].to(device)
                            masks = batch_data["label"].to(device)
                            
                            with torch.amp.autocast('cuda'):
                                # Use overlap=0.25 and sw_batch_size=4 for faster validation
                                val_outputs = sliding_window_inference(images, (IMG_SIZE, IMG_SIZE, IMG_SIZE), 3, model, overlap=0.25)
                                loss = criterion(val_outputs, masks)
                            
                            val_loss += loss.item()
                            
                            # Compute Dice Metric
                            val_outputs = [post_trans(i) for i in decollate_batch(val_outputs)]
                            val_labels = decollate_batch(masks)
                            dice_metric(y_pred=val_outputs, y=val_labels)
                            dice_metric_batch(y_pred=val_outputs, y=val_labels)
                
                avg_val_loss = val_loss / len(val_loader)
                
                # Aggregate Metrics
                mean_val_dice = dice_metric.aggregate().item()
                metric_batch = dice_metric_batch.aggregate()
                
                # Metric per class (TC, WT, ET)
                metric_tc = metric_batch[0].item()
                metric_wt = metric_batch[1].item()
                metric_et = metric_batch[2].item()
                
                dice_metric.reset()
                dice_metric_batch.reset()
                
                # Log Metrics
                experiment.log_metric("epoch_val_loss", avg_val_loss, step=epoch)
                experiment.log_metric("val_mean_dice", mean_val_dice, step=epoch)
                experiment.log_metric("val_dice_TC", metric_tc, step=epoch)
                experiment.log_metric("val_dice_WT", metric_wt, step=epoch)
                experiment.log_metric("val_dice_ET", metric_et, step=epoch)
                
                print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Mean Dice: {mean_val_dice:.4f}")
                print(f"Dice Classes -> TC: {metric_tc:.4f}, WT: {metric_wt:.4f}, ET: {metric_et:.4f}")
                
                # === SAVE BEST MODEL (Based on Dice) ===
                if mean_val_dice > best_val_dice:
                    best_val_dice = mean_val_dice
                    epochs_no_improve = 0
                    torch.save(model.state_dict(), save_path)
                    experiment.log_model("UNet_Best", save_path)
                    print(f"Validation Dice improved. Model saved to {save_path}")
                else:
                    epochs_no_improve += 1
                    print(f"No improvement for {epochs_no_improve} checks.")
                    
                # === EARLY STOPPING ===
                if epochs_no_improve >= patience:
                    print("Early stopping triggered!")
                    break
            else:
                print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f} | Validation Skipped")
            
            # === SCHEDULER ===
            scheduler.step()
                
            gc.collect()
            torch.cuda.empty_cache()
            
train_loader, val_loader = get_loader(BATCH_SIZE, DATASET_ROOT, roi=(IMG_SIZE, IMG_SIZE, IMG_SIZE))
train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, NUM_EPOCHS, patience=5, val_interval=3)

## 6. Visualization
Visualize a sample prediction.


In [ ]:
# Visualize and Log to Comet
def visualize_prediction(model, dataset, index=0):
    model.eval()
    data = dataset[index]
    image = data['image']
    mask = data['label']
    
    # Prepare input
    input_tensor = image.unsqueeze(0).to(device)
    with torch.no_grad():
        output = sliding_window_inference(input_tensor, (IMG_SIZE, IMG_SIZE, IMG_SIZE), 3, model, overlap=0.5)
        # Output is (1, 3, D, H, W). Use sigmoid > 0.5 for regions
        prediction = (torch.sigmoid(output) > 0.5).float().cpu().numpy()[0]
        
    # Visualize Middle Slice
    # Find slice with maximum label area (Enhancing Tumor)
    if mask.sum() > 0:
        # Try Enhancing Tumor (channel 2)
        slice_areas = mask[2].sum(dim=(0, 1))
        if slice_areas.max() > 0:
            slice_idx = torch.argmax(slice_areas).item()
        else:
            # Any tumor class
            slice_areas = mask.sum(dim=0).sum(dim=(0, 1))
            if slice_areas.max() > 0:
                slice_idx = torch.argmax(slice_areas).item()
            else:
                slice_idx = image.shape[-1] // 2
    else:
        slice_idx = image.shape[-1] // 2
    print(f"Visualizing Slice: {slice_idx}")
    
    fig, ax = plt.subplots(1, 4, figsize=(20, 5))
    
    # Show FLAIR channel (channel 0)
    ax[0].imshow(image[0, :, :, slice_idx].cpu(), cmap='gray')
    ax[0].set_title("Input (FLAIR)")
    
    # Show T1ce channel (channel 2)
    ax[1].imshow(image[2, :, :, slice_idx].cpu(), cmap='gray')
    ax[1].set_title("Input (T1ce)")

    # Show Ground Truth (Enhancing Tumor - channel 2)
    ax[2].imshow(mask[2, :, :, slice_idx].cpu(), cmap='gray')
    ax[2].set_title("Ground Truth (ET)")
    
    # Show Prediction (Enhancing Tumor - channel 2)
    ax[3].imshow(prediction[2, :, :, slice_idx], cmap='gray')
    ax[3].set_title("Prediction (ET)")
    
    # Save figure to log
    plt.savefig("prediction_sample.png")
    experiment.log_image("prediction_sample.png", name=f"Prediction Sample Index {index}")
    
    plt.show()

# Visualize
visualize_prediction(model, val_loader.dataset, index=0)

# Inference

In [ ]:
# === SUBMISSION PIPELINE ===
def rle_encoding(x):
    '''
    x: numpy array of shape (height, width), 1 - mask, 0 - background
    Returns run length as list
    '''
    dots = np.where(x.flatten() == 1)[0] # .flatten() gives row-major flattening
    if len(dots) == 0:
        return ""
    
    run_lengths = []
    prev = -2
    for b in dots:
        if (b > prev + 1):
            run_lengths.extend((b + 1, 0))
        run_lengths[-1] += 1
        prev = b
        
    return " ".join([str(x) for x in run_lengths])

def run_inference_and_submit(model, test_dir, output_file="submission.csv"):
    test_transform = transforms.Compose([
        transforms.LoadImaged(keys=["image"]),
        transforms.EnsureChannelFirstd(keys=["image"]),
        transforms.NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    ])
    
    test_dataset = BraTSDataset(root_dir="", transform=test_transform, mode='test', test_dir=test_dir)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    
    model.eval()
    
    results = []
    
    with open(output_file, 'w') as f:
        f.write("id,rle\n")
        
        with torch.no_grad():
            for batch_data in tqdm(test_loader, desc="Inference"):
                images = batch_data["image"].to(device)
                case_id = batch_data["id"][0]
                
                # Inference
                outputs = sliding_window_inference(images, (IMG_SIZE, IMG_SIZE, IMG_SIZE), 3, model, overlap=0.5)
                
                # Output channels: 0: TC, 1: WT, 2: ET
                probs = torch.sigmoid(outputs)
                preds = (probs > 0.5).float().cpu().numpy()[0] # (3, D, H, W)
                
                # Reconstruct Classes for Submission
                # Class 1 (NCR): TC (0) - ET (2)
                # Class 2 (ED): WT (1) - TC (0)
                # Class 4 (ET): ET (2)
                
                tc = preds[0]
                wt = preds[1]
                et = preds[2]
                
                label_1 = (tc - et)
                label_1[label_1 < 0] = 0 # Safety clip
                
                label_2 = (wt - tc)
                label_2[label_2 < 0] = 0
                
                label_4 = et
                
                # Encode and Write
                # 1: NCR
                rle_1 = rle_encoding(label_1)
                f.write(f"{case_id}_1,{rle_1}\n")
                
                # 2: ED
                rle_2 = rle_encoding(label_2)
                f.write(f"{case_id}_2,{rle_2}\n")
                
                # 4: ET
                rle_4 = rle_encoding(label_4)
                f.write(f"{case_id}_4,{rle_4}\n")
                
    print(f"Submission saved to {output_file}")

# model_path = "/kaggle/input/brain-tumor-segmentation/best_model.pth"
# model.load_state_dict(torch.load(model_path, map_location=device))
# print(f"Loaded model from {model_path}")

# Run Submission Pipeline
TEST_DIR = "/kaggle/input/instant-odc-ai-hackathon/test"
run_inference_and_submit(model, TEST_DIR)

In [ ]:
experiment.end()